# Q-Learning vs Dyna-Q: Comprehensive Comparison

This notebook provides a complete side-by-side comparison of **Q-Learning** and **Dyna-Q** on the same random grid world environment. Includes:
- Environment visualization
- **Learning curves**: reward, success rate, steps over episodes
- **Q-value heatmaps**: comparing learned state values
- **Policy arrows**: visualizing learned greedy policies
- **Greedy paths**: final paths taken by each agent
- **Dynamic animation**: path discovery over training for both agents side-by-side
- **Statistical comparison**: metrics table with statistical analysis
- **Multiple trials**: running multiple trials for robust comparison

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import time
from collections import defaultdict

from src.environment import GridWorld
from src.agents.q_learning import QLearningAgent
from src.agents.dyna_q import DynaQAgent
from src.trainer import train_agent, extract_greedy_path, run_multiple_trials
from src.metrics import compute_metrics, compute_multi_trial_metrics, format_metrics_table

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 8)
plt.rcParams['figure.dpi'] = 100

## 1. Environment Setup

In [ ]:
# Environment Configuration
GRID_SIZE = 15
OBSTACLE_DENSITY = 0.25
SEED = 42

# Training Configuration
NUM_EPISODES = 500
MAX_STEPS = 500
ALPHA = 0.1
GAMMA = 0.95
EPSILON_DECAY = 0.995
N_PLANNING = 5

# Create environment
env = GridWorld(size=GRID_SIZE, obstacle_density=OBSTACLE_DENSITY, seed=SEED)
optimal_length = env.get_optimal_path_length()

print("=" * 60)
print("ENVIRONMENT CONFIGURATION")
print("=" * 60)
print(f"Grid Size:            {env.size}x{env.size}")
print(f"Total States:        {env.num_states}")
print(f"Total Actions:       {env.num_actions}")
print(f"Free Cells:          {np.sum(env.grid == 0) + 2}")
print(f"Obstacles:           {np.sum(env.grid == 1)}")
print(f"Obstacle Density:    {OBSTACLE_DENSITY*100}%")
print(f"Start Position:      {env.start}")
print(f"Goal Position:       {env.goal}")
print(f"Optimal Path Length: {optimal_length} steps")
print("=" * 60)

In [ ]:
def show_grid(env, path=None, title="Grid World", ax=None, show=True):
    """Render grid with optional path overlay."""
    size = env.size
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    
    display_grid = np.zeros((size, size, 3))
    for r in range(size):
        for c in range(size):
            if env.grid[r, c] == 1:       # obstacle
                display_grid[r, c] = [0.2, 0.2, 0.2]
            elif env.grid[r, c] == 2:     # start
                display_grid[r, c] = [0.2, 0.8, 0.2]
            elif env.grid[r, c] == 3:     # goal
                display_grid[r, c] = [0.2, 0.4, 1.0]
            else:                          # free
                display_grid[r, c] = [1.0, 1.0, 1.0]
    
    # Overlay path with gradient coloring
    if path and len(path) > 1:
        for i, (r, c) in enumerate(path):
            if env.grid[r, c] == 0:  # only color free cells
                t = i / max(len(path) - 1, 1)
                display_grid[r, c] = [1.0 - 0.6*t, 1.0 - 0.3*t, 0.8 - 0.8*t]
    
    ax.imshow(display_grid, interpolation='nearest')
    
    # Grid lines
    for i in range(size + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.5, alpha=0.3)
        ax.axvline(i - 0.5, color='gray', linewidth=0.5, alpha=0.3)
    
    # Path arrows
    if path and len(path) > 1:
        for i in range(len(path) - 1):
            ax.annotate('', xy=(path[i+1][1], path[i+1][0]),
                        xytext=(path[i][1], path[i][0]),
                        arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    
    # Markers
    ax.plot(env.start[1], env.start[0], 'g*', markersize=15, label='Start')
    ax.plot(env.goal[1], env.goal[0], 'b*', markersize=15, label='Goal')
    if path:
        ax.plot(path[-1][1], path[-1][0], 'ro', markersize=10, label='End')
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    
    return ax

fig, ax = plt.subplots(figsize=(8, 8))
show_grid(env, title=f"Random Grid World ({GRID_SIZE}x{GRID_SIZE})", ax=ax)
plt.tight_layout()
plt.show()

## 2. Train Both Agents

In [ ]:
# Train Q-Learning Agent
print("Training Q-Learning Agent...")
agent_ql = QLearningAgent(
    num_states=env.num_states,
    num_actions=env.num_actions,
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon_decay=EPSILON_DECAY
)

start = time.time()
results_ql = train_agent(env, agent_ql, num_episodes=NUM_EPISODES, 
                         max_steps=MAX_STEPS, seed=SEED)
time_ql = time.time() - start

# Train Dyna-Q Agent
print("Training Dyna-Q Agent...")
agent_dq = DynaQAgent(
    num_states=env.num_states,
    num_actions=env.num_actions,
    n_planning=N_PLANNING,
    alpha=ALPHA,
    gamma=GAMMA,
    epsilon_decay=EPSILON_DECAY
)

start = time.time()
results_dq = train_agent(env, agent_dq, num_episodes=NUM_EPISODES, 
                         max_steps=MAX_STEPS, seed=SEED)
time_dq = time.time() - start

print(f"\nQ-Learning training time: {time_ql:.2f}s")
print(f"Dyna-Q training time: {time_dq:.2f}s")

## 3. Learning Curves Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

window = 50

# 1. Raw Rewards
ax = axes[0, 0]
ax.plot(results_ql['rewards'], alpha=0.2, color='blue')
ax.plot(results_dq['rewards'], alpha=0.2, color='purple')
smooth_ql = np.convolve(results_ql['rewards'], np.ones(window)/window, mode='valid')
smooth_dq = np.convolve(results_dq['rewards'], np.ones(window)/window, mode='valid')
ax.plot(range(window-1, NUM_EPISODES), smooth_ql, label='Q-Learning', linewidth=2, color='blue')
ax.plot(range(window-1, NUM_EPISODES), smooth_dq, label='Dyna-Q', linewidth=2, color='purple')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Reward per Episode (Smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Success Rate
ax = axes[0, 1]
sr_ql = np.convolve(results_ql['success'], np.ones(window)/window, mode='valid')
sr_dq = np.convolve(results_dq['success'], np.ones(window)/window, mode='valid')
ax.plot(range(window-1, NUM_EPISODES), sr_ql * 100, label='Q-Learning', linewidth=2, color='blue')
ax.plot(range(window-1, NUM_EPISODES), sr_dq * 100, label='Dyna-Q', linewidth=2, color='purple')
ax.axhline(y=90, color='green', linestyle='--', alpha=0.7, label='90% threshold')
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Rolling Success Rate (window=50)')
ax.legend()
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# 3. Steps per Episode
ax = axes[0, 2]
ax.plot(results_ql['steps'], alpha=0.2, color='blue')
ax.plot(results_dq['steps'], alpha=0.2, color='purple')
step_ql = np.convolve(results_ql['steps'], np.ones(window)/window, mode='valid')
step_dq = np.convolve(results_dq['steps'], np.ones(window)/window, mode='valid')
ax.plot(range(window-1, NUM_EPISODES), step_ql, label='Q-Learning', linewidth=2, color='blue')
ax.plot(range(window-1, NUM_EPISODES), step_dq, label='Dyna-Q', linewidth=2, color='purple')
ax.axhline(y=optimal_length, color='green', linestyle='--', label=f'Optimal ({optimal_length})')
ax.set_xlabel('Episode')
ax.set_ylabel('Steps')
ax.set_title('Steps per Episode (Smoothed)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Cumulative Reward
ax = axes[1, 0]
ax.plot(np.cumsum(results_ql['rewards']), label='Q-Learning', linewidth=2, color='blue')
ax.plot(np.cumsum(results_dq['rewards']), label='Dyna-Q', linewidth=2, color='purple')
ax.set_xlabel('Episode')
ax.set_ylabel('Cumulative Reward')
ax.set_title('Cumulative Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Epsilon Decay
ax = axes[1, 1]
ax.plot(results_ql['epsilons'], label='Q-Learning', linewidth=2, color='blue')
ax.plot(results_dq['epsilons'], label='Dyna-Q', linewidth=2, color='purple', linestyle='--')
ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon')
ax.set_title('Exploration Rate (ε)')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Final Performance Distribution (Last 50 episodes)
ax = axes[1, 2]
last_50_ql = results_ql['rewards'][-50:]
last_50_dq = results_dq['rewards'][-50:]
bp = ax.boxplot([last_50_ql, last_50_dq], labels=['Q-Learning', 'Dyna-Q'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lavender')
ax.set_ylabel('Episode Reward')
ax.set_title('Final Performance Distribution (Last 50 Episodes)')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Q-Learning vs Dyna-Q: Learning Curves Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Q-Value Heatmaps Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, agent, title, cmap in [
    (axes[0], agent_ql, 'Q-Learning', 'Blues'),
    (axes[1], agent_dq, 'Dyna-Q', 'Purples')
]:
    state_values = np.max(agent.q_table, axis=1).reshape(env.size, env.size)
    
    im = ax.imshow(state_values, cmap=cmap, interpolation='nearest')
    
    # Mask obstacles
    for r in range(env.size):
        for c in range(env.size):
            if env.grid[r, c] == 1:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='black'))
    
    ax.plot(env.start[1], env.start[0], 'g*', markersize=15)
    ax.plot(env.goal[1], env.goal[0], 'b*', markersize=15)
    
    plt.colorbar(im, ax=ax, label='Max Q-value', shrink=0.8)
    ax.set_title(f'{title}: State Value Heatmap', fontweight='bold', fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Q-Value Heatmaps: State Values V(s) = max_a Q(s,a)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Learned Policy Arrows Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

arrow_map = {0: (0, -0.35), 1: (0, 0.35), 2: (-0.35, 0), 3: (0.35, 0)}

for ax, agent, title, color in [
    (axes[0], agent_ql, 'Q-Learning', 'blue'),
    (axes[1], agent_dq, 'Dyna-Q', 'purple')
]:
    # Background
    display_grid = np.ones((env.size, env.size, 3))
    for r in range(env.size):
        for c in range(env.size):
            if env.grid[r, c] == 1:
                display_grid[r, c] = [0.3, 0.3, 0.3]
    ax.imshow(display_grid, interpolation='nearest')
    
    # Policy arrows
    for s in range(env.num_states):
        pos = env.state_to_pos(s)
        if env.grid[pos] == 1:
            continue
        best_action = np.argmax(agent.q_table[s])
        dx, dy = arrow_map[best_action]
        ax.arrow(pos[1], pos[0], dx, dy, head_width=0.15, head_length=0.1,
                 fc=color, ec=color, alpha=0.7)
    
    # Grid
    for i in range(env.size + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.3, alpha=0.3)
        ax.axvline(i - 0.5, color='gray', linewidth=0.3, alpha=0.3)
    
    ax.plot(env.start[1], env.start[0], 'g*', markersize=15)
    ax.plot(env.goal[1], env.goal[0], 'b*', markersize=15)
    ax.set_title(f'{title}: Greedy Policy', fontweight='bold', fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Learned Policies: Greedy Actions from Q-Table', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Final Greedy Paths Comparison

In [ ]:
# Extract greedy paths
path_ql = extract_greedy_path(env, agent_ql)
path_dq = extract_greedy_path(env, agent_dq)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, path, title, agent_name in [
    (axes[0], path_ql, 'Q-Learning', 'Q-Learning'),
    (axes[1], path_dq, 'Dyna-Q', 'Dyna-Q')
]:
    show_grid(env, path=path, title=f'{title} Greedy Path', ax=ax)
    
    success = path[-1] == env.goal
    steps = len(path) - 1
    optimality = steps / optimal_length if optimal_length > 0 else float('inf')
    
    info_text = f"Steps: {steps} | Optimal: {optimal_length} | Ratio: {optimality:.2f}x"
    if success:
        info_text += " | ✓ GOAL REACHED"
    else:
        info_text += " | ✗ FAILED"
    
    ax.text(0.5, -0.08, info_text, transform=ax.transAxes, 
            ha='center', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Final Greedy Paths: Testing Learned Policies (ε=0)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Dynamic Path Discovery Animation

Watch both agents discover their paths over training. Shows the best path found up to each checkpoint.

In [ ]:
# Re-train with frequent snapshots for animation
env_anim = GridWorld(size=GRID_SIZE, obstacle_density=OBSTACLE_DENSITY, seed=SEED)

agent_ql_anim = QLearningAgent(
    num_states=env_anim.num_states, num_actions=env_anim.num_actions,
    alpha=ALPHA, gamma=GAMMA, epsilon_decay=EPSILON_DECAY
)
agent_dq_anim = DynaQAgent(
    num_states=env_anim.num_states, num_actions=env_anim.num_actions,
    n_planning=N_PLANNING, alpha=ALPHA, gamma=GAMMA, epsilon_decay=EPSILON_DECAY
)

SNAP_EVERY = 20
snapshots_ql = []
snapshots_dq = []

np.random.seed(SEED)
agent_ql_anim.reset()
agent_dq_anim.reset()

best_path_ql, best_steps_ql = None, float('inf')
best_path_dq, best_steps_dq = None, float('inf')

all_rewards_ql, all_rewards_dq = [], []
all_success_ql, all_success_dq = [], []

for ep in range(NUM_EPISODES):
    # Q-Learning episode
    state = env_anim.reset()
    path = [env_anim.agent_pos]
    total_r = 0.0
    for s in range(MAX_STEPS):
        a = agent_ql_anim.select_action(state)
        ns, r, done = env_anim.step(a)
        agent_ql_anim.update(state, a, r, ns, done)
        state = ns
        total_r += r
        path.append(env_anim.agent_pos)
        if done: break
    agent_ql_anim.decay_epsilon()
    success = (env_anim.agent_pos == env_anim.goal)
    all_rewards_ql.append(total_r)
    all_success_ql.append(success)
    if success and len(path)-1 < best_steps_ql:
        best_steps_ql = len(path) - 1
        best_path_ql = path.copy()
    
    # Dyna-Q episode
    state = env_anim.reset()
    path = [env_anim.agent_pos]
    total_r = 0.0
    for s in range(MAX_STEPS):
        a = agent_dq_anim.select_action(state)
        ns, r, done = env_anim.step(a)
        agent_dq_anim.update(state, a, r, ns, done)
        state = ns
        total_r += r
        path.append(env_anim.agent_pos)
        if done: break
    agent_dq_anim.decay_epsilon()
    success = (env_anim.agent_pos == env_anim.goal)
    all_rewards_dq.append(total_r)
    all_success_dq.append(success)
    if success and len(path)-1 < best_steps_dq:
        best_steps_dq = len(path) - 1
        best_path_dq = path.copy()
    
    # Snapshot
    if (ep + 1) % SNAP_EVERY == 0:
        w = min(50, len(all_success_ql))
        sr_ql = np.mean(all_success_ql[-w:]) * 100
        sr_dq = np.mean(all_success_dq[-w:]) * 100
        snapshots_ql.append({'ep': ep+1, 'path': best_path_ql.copy(), 
                           'steps': best_steps_ql, 'sr': sr_ql})
        snapshots_dq.append({'ep': ep+1, 'path': best_path_dq.copy(), 
                           'steps': best_steps_dq, 'sr': sr_dq})

n_snaps = min(len(snapshots_ql), 16)
indices = np.linspace(0, len(snapshots_ql)-1, n_snaps, dtype=int)

cols = 4
rows = (n_snaps + cols - 1) // cols
fig, axes = plt.subplots(rows * 2, cols, figsize=(4*cols, 4*rows))

for i, ax_idx in enumerate(indices):
    # Q-Learning (top row)
    ax_ql = axes[i // cols * 2, i % cols]
    snap = snapshots_ql[ax_idx]
    show_grid(env_anim, path=snap['path'], title=f"QL: Ep {snap['ep']}", ax=ax_ql, show=False)
    ax_ql.set_xlabel(f"Steps:{snap['steps']} SR:{snap['sr']:.0f}%", fontsize=9)
    
    # Dyna-Q (bottom row)
    ax_dq = axes[i // cols * 2 + 1, i % cols]
    snap = snapshots_dq[ax_idx]
    show_grid(env_anim, path=snap['path'], title=f"DQ: Ep {snap['ep']}", ax=ax_dq, show=False)
    ax_dq.set_xlabel(f"Steps:{snap['steps']} SR:{snap['sr']:.0f}%", fontsize=9)

plt.suptitle('Path Discovery Over Training: Q-Learning (top) vs Dyna-Q (bottom)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Statistical Metrics Comparison

In [ ]:
metrics_ql = compute_metrics(results_ql, optimal_length)
metrics_dq = compute_metrics(results_dq, optimal_length)

print("=" * 70)
print("METRICS COMPARISON: Q-LEARNING vs Dyna-Q")
print("=" * 70)
print(format_metrics_table(metrics_ql, metrics_dq))
print("=" * 70)

In [ ]:
# Detailed bar chart comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

metrics_names = [
    ('Final Success Rate', 'final_success_rate', '%', 100),
    ('Best Path Steps', 'best_path_steps', '', 1),
    ('Final Avg Reward', 'final_avg_reward', '', 1),
    ('Final Avg Steps', 'final_avg_steps', '', 1),
    ('Convergence Episode', 'convergence_episode', '', 1),
    ('Training Time', 'training_time', 's', 1),
]

for idx, (name, key, unit, scale) in enumerate(metrics_names):
    ax = axes[idx // 3, idx % 3]
    v_ql = metrics_ql.get(key, 0)
    v_dq = metrics_dq.get(key, 0)
    
    bars = ax.bar(['Q-Learning', 'Dyna-Q'], [v_ql*scale, v_dq*scale], 
                  color=['steelblue', 'mediumpurple'], edgecolor='black')
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_ylabel(unit if unit else name)
    
    # Add value labels
    for bar, val in zip(bars, [v_ql*scale, v_dq*scale]):
        height = bar.get_height()
        ax.annotate(f'{val:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)
    
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Performance Metrics Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Multiple Trials Comparison (Robust Analysis)

Running multiple independent trials to get statistically significant results.

In [ ]:
print("Running multiple trials for statistical robustness...")
print("(This may take a minute...)\n")

NUM_TRIALS = 10

# Multiple trials for Q-Learning
def make_ql():
    return QLearningAgent(env.num_states, env.num_actions, alpha=ALPHA, gamma=GAMMA, 
                          epsilon_decay=EPSILON_DECAY)

def make_dq():
    return DynaQAgent(env.num_states, env.num_actions, n_planning=N_PLANNING,
                      alpha=ALPHA, gamma=GAMMA, epsilon_decay=EPSILON_DECAY)

trial_results_ql = run_multiple_trials(env, make_ql, num_trials=NUM_TRIALS, 
                                         num_episodes=NUM_EPISODES, base_seed=42)
trial_results_dq = run_multiple_trials(env, make_dq, num_trials=NUM_TRIALS, 
                                         num_episodes=NUM_EPISODES, base_seed=42)

multi_metrics_ql = compute_multi_trial_metrics(trial_results_ql, optimal_length)
multi_metrics_dq = compute_multi_trial_metrics(trial_results_dq, optimal_length)

print(f"Completed {NUM_TRIALS} trials for each algorithm.")

In [ ]:
# Aggregate results across trials
print("\n" + "=" * 70)
print("MULTI-TRIAL STATISTICS (10 trials)")
print("=" * 70)

print(f"\n{'Metric':<30} {'Q-Learning':>15} {'Dyna-Q':>15} {'Better':>10}")
print("-" * 70)

comparisons = [
    ('Avg Success Rate', 'avg_final_success_rate', 'higher'),
    ('Avg Best Steps', 'avg_best_steps', 'lower'),
    ('Avg Convergence Ep', 'avg_convergence_episode', 'lower'),
    ('Avg Training Time', 'avg_training_time', 'lower'),
]

for name, key, better in comparisons:
    v_ql = multi_metrics_ql[key]
    v_dq = multi_metrics_dq[key]
    
    if better == 'higher':
        winner = 'Q-Learn' if v_ql > v_dq else 'Dyna-Q' if v_dq > v_ql else 'Tie'
    else:
        winner = 'Q-Learn' if v_ql < v_dq else 'Dyna-Q' if v_dq < v_ql else 'Tie'
    
    print(f"{name:<30} {v_ql:>15.2f} {v_dq:>15.2f} {winner:>10}")

print("-" * 70)
print(f"Success Rate Std Dev:     QL={multi_metrics_ql['std_final_success_rate']:.3f}  "
      f"DQ={multi_metrics_dq['std_final_success_rate']:.3f}")
print(f"Convergence Std Dev:      QL={multi_metrics_ql['std_convergence_episode']:.1f}  "
      f"DQ={multi_metrics_dq['std_convergence_episode']:.1f}")
print("=" * 70)

In [ ]:
# Visualize multi-trial distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

success_rates_ql = [m['final_success_rate'] * 100 for m in multi_metrics_ql['per_trial']]
success_rates_dq = [m['final_success_rate'] * 100 for m in multi_metrics_dq['per_trial']]
best_steps_ql = [m['best_steps'] for m in multi_metrics_ql['per_trial'] if m['best_steps'] > 0]
best_steps_dq = [m['best_steps'] for m in multi_metrics_dq['per_trial'] if m['best_steps'] > 0]
conv_ep_ql = [m['convergence_episode'] for m in multi_metrics_ql['per_trial']]
conv_ep_dq = [m['convergence_episode'] for m in multi_metrics_dq['per_trial']]
train_time_ql = [m['training_time'] for m in multi_metrics_ql['per_trial']]
train_time_dq = [m['training_time'] for m in multi_metrics_dq['per_trial']]

# 1. Success Rate
ax = axes[0, 0]
bp = ax.boxplot([success_rates_ql, success_rates_dq], labels=['Q-Learning', 'Dyna-Q'], 
                patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lavender')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Success Rate Distribution (10 trials)')
ax.grid(True, alpha=0.3, axis='y')

# 2. Best Steps
ax = axes[0, 1]
bp = ax.boxplot([best_steps_ql, best_steps_dq], labels=['Q-Learning', 'Dyna-Q'],
                patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lavender')
ax.axhline(y=optimal_length, color='green', linestyle='--', label=f'Optimal ({optimal_length})')
ax.set_ylabel('Best Path Steps')
ax.set_title('Best Path Steps Distribution')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 3. Convergence Episode
ax = axes[1, 0]
bp = ax.boxplot([conv_ep_ql, conv_ep_dq], labels=['Q-Learning', 'Dyna-Q'],
                patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lavender')
ax.set_ylabel('Episode')
ax.set_title('Convergence Episode Distribution')
ax.grid(True, alpha=0.3, axis='y')

# 4. Training Time
ax = axes[1, 1]
bp = ax.boxplot([train_time_ql, train_time_dq], labels=['Q-Learning', 'Dyna-Q'],
                patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lavender')
ax.set_ylabel('Time (seconds)')
ax.set_title('Training Time Distribution')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Multi-Trial Performance Distribution (10 independent trials)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Summary & Conclusions

In [ ]:
print("\n" + "=" * 70)
print("FINAL SUMMARY: Q-LEARNING vs Dyna-Q")
print("=" * 70)

print(f"""
Environment:
  - Grid Size: {GRID_SIZE}x{GRID_SIZE} ({env.num_states} states)
  - Obstacles: {np.sum(env.grid == 1)} ({OBSTACLE_DENSITY*100}% density)
  - Optimal Path: {optimal_length} steps

Training:
  - Episodes: {NUM_EPISODES}
  - Learning Rate: {ALPHA}
  - Discount Factor: {GAMMA}
  - Epsilon Decay: {EPSILON_DECAY}
  - Dyna-Q Planning Steps: {N_PLANNING}

Results (Single Trial):
  Q-Learning:
    - Best Path: {metrics_ql['best_path_steps']} steps
    - Final Success Rate: {metrics_ql['final_success_rate']*100:.1f}%
    - Convergence: Episode {metrics_ql['convergence_episode']}
  
  Dyna-Q:
    - Best Path: {metrics_dq['best_path_steps']} steps
    - Final Success Rate: {metrics_dq['final_success_rate']*100:.1f}%
    - Convergence: Episode {metrics_dq['convergence_episode']}

Results ({NUM_TRIALS} Trials Average):
  Q-Learning:
    - Avg Success Rate: {multi_metrics_ql['avg_final_success_rate']*100:.1f}%
    - Avg Best Steps: {multi_metrics_ql['avg_best_steps']:.1f}
  
  Dyna-Q:
    - Avg Success Rate: {multi_metrics_dq['avg_final_success_rate']*100:.1f}%
    - Avg Best Steps: {multi_metrics_dq['avg_best_steps']:.1f}

Key Insights:
  - Dyna-Q uses planning (model-based) to accelerate learning
  - Dyna-Q typically converges faster with fewer real environment steps
  - Both algorithms can find optimal or near-optimal paths
""")
print("=" * 70)